In [0]:
%pip install azure-storage-blob pyarrow
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import io
import pandas as pd
from azure.storage.blob import ContainerClient

# ---- Settings ----
ACCOUNT_URL = "https://azureopendatastorage.blob.core.windows.net"
CONTAINER   = "nyctlc"                      # public NYC taxi dataset
PREFIX      = "yellow/puYear=2018/puMonth=1/"
NUM_FILES   = 3                             # how many parquet files to load
ROWS_PER_FILE = 50000                       # cap rows per file to keep memory low
TABLE_NAME  = "workspace.default.azure_taxi_sample"

# ---- 1. Connect (anonymous, public container) ----
cc = ContainerClient(ACCOUNT_URL, CONTAINER)

# ---- 2. List only real parquet files (skip _SUCCESS / _committed markers) ----
parquet_files = [
    b for b in cc.list_blobs(name_starts_with=PREFIX)
    if b.name.endswith(".parquet")
]
print(f"{len(parquet_files)} parquet files found")

# ---- 3. Download and combine ----
frames = []
for b in parquet_files[:NUM_FILES]:
    print(f"Reading {b.name.split('/')[-1][:40]}... ({round(b.size/1e6, 1)} MB)")
    data = cc.download_blob(b.name).readall()
    frames.append(pd.read_parquet(io.BytesIO(data)).head(ROWS_PER_FILE))

pdf = pd.concat(frames, ignore_index=True)
print(f"Combined: {len(pdf)} rows")

# ---- 4. Convert to Spark and save as a Unity Catalog table ----
df = spark.createDataFrame(pdf)
df.write.mode("overwrite").saveAsTable(TABLE_NAME)
print(f"Saved to {TABLE_NAME}")

20 parquet files found
Reading part-00000-tid-8898858832658823408-a1de8... (11.7 MB)
Reading part-00001-tid-8898858832658823408-a1de8... (10.2 MB)
Reading part-00002-tid-8898858832658823408-a1de8... (10.2 MB)
Combined: 150000 rows
Saved to workspace.default.azure_taxi_sample


In [0]:
df = spark.table("workspace.default.azure_taxi_sample")
print(df.count(), "rows")
df.printSchema()
display(df.limit(10))

150000 rows
root
 |-- vendorID: string (nullable = true)
 |-- tpepPickupDateTime: timestamp (nullable = true)
 |-- tpepDropoffDateTime: timestamp (nullable = true)
 |-- passengerCount: integer (nullable = true)
 |-- tripDistance: double (nullable = true)
 |-- puLocationId: string (nullable = true)
 |-- doLocationId: string (nullable = true)
 |-- startLon: double (nullable = true)
 |-- startLat: double (nullable = true)
 |-- endLon: double (nullable = true)
 |-- endLat: double (nullable = true)
 |-- rateCodeId: integer (nullable = true)
 |-- storeAndFwdFlag: string (nullable = true)
 |-- paymentType: string (nullable = true)
 |-- fareAmount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mtaTax: double (nullable = true)
 |-- improvementSurcharge: string (nullable = true)
 |-- tipAmount: double (nullable = true)
 |-- tollsAmount: double (nullable = true)
 |-- totalAmount: double (nullable = true)



vendorID,tpepPickupDateTime,tpepDropoffDateTime,passengerCount,tripDistance,puLocationId,doLocationId,startLon,startLat,endLon,endLat,rateCodeId,storeAndFwdFlag,paymentType,fareAmount,extra,mtaTax,improvementSurcharge,tipAmount,tollsAmount,totalAmount
2,2018-01-01T14:30:35.000Z,2018-01-01T14:38:26.000Z,5,1.37,239,163,null,null,null,null,1,N,1,7.5,0.0,0.5,0.3,0.0,0.0,8.3
1,2018-01-01T01:25:31.000Z,2018-01-01T01:28:33.000Z,1,0.2,264,264,null,null,null,null,1,N,2,4.0,0.5,0.5,0.3,0.0,0.0,5.3
2,2018-01-01T02:01:32.000Z,2018-01-01T02:03:38.000Z,1,0.33,231,125,null,null,null,null,1,N,1,3.5,0.5,0.5,0.3,0.96,0.0,5.76
2,2018-01-01T00:30:44.000Z,2018-01-01T00:45:03.000Z,2,3.03,151,262,null,null,null,null,1,N,1,12.5,0.5,0.5,0.3,0.0,0.0,13.8
2,2018-01-01T02:00:51.000Z,2018-01-02T01:49:54.000Z,2,2.13,239,48,null,null,null,null,1,N,1,11.5,0.5,0.5,0.3,1.92,0.0,14.72
2,2018-01-01T08:12:28.000Z,2018-01-01T08:13:27.000Z,2,0.3,129,129,null,null,null,null,1,N,2,3.0,0.0,0.5,0.3,0.0,0.0,3.8
2,2018-01-01T00:50:03.000Z,2018-01-01T01:22:20.000Z,6,11.5,263,62,null,null,null,null,1,N,1,36.0,0.5,0.5,0.3,9.32,0.0,46.62
2,2018-01-01T12:04:48.000Z,2018-01-01T12:14:09.000Z,2,1.32,100,162,null,null,null,null,1,N,1,8.0,0.0,0.5,0.3,2.64,0.0,11.44
2,2018-01-01T10:36:58.000Z,2018-01-01T10:47:04.000Z,1,1.18,163,43,null,null,null,null,1,N,1,7.5,0.0,0.5,0.3,2.49,0.0,10.79
2,2018-01-01T01:09:38.000Z,2018-01-01T01:39:59.000Z,1,5.06,80,114,null,null,null,null,1,N,1,23.0,0.5,0.5,0.3,4.86,0.0,29.16


In [0]:
%sql
SELECT passengerCount,
       COUNT(*) AS trips,
       ROUND(AVG(fareAmount), 2) AS avg_fare,
       ROUND(AVG(tipAmount), 2) AS avg_tip
FROM workspace.default.azure_taxi_sample
WHERE fareAmount > 0
GROUP BY passengerCount
ORDER BY passengerCount;


passengerCount,trips,avg_fare,avg_tip
0,1014,12.6,1.71
1,106734,12.34,1.8
2,21866,13.22,1.81
3,6025,13.32,1.75
4,2953,13.61,1.64
5,6990,12.36,1.79
6,4302,12.22,1.78
7,1,7.3,1.52
